In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import time

import numpy as np
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.termination.collection import TerminationCollection

from sudoku.operators import (
    EPLSurvival,
    LocalSearchRepair,
    LocalSearchRepairV2,
    MySampling,
    RowCrossover,
    SwapReinitMutation,
    ZeroFunctionValueTermination,
)
from sudoku.problem import SudokuProblem

In [14]:
hard_no_77 = np.array(
    [
        [5, 0, 0, 0, 0, 0, 0, 0, 9],
        [9, 0, 0, 8, 0, 5, 0, 0, 6],
        [3, 0, 0, 9, 0, 7, 0, 0, 5],
        [0, 0, 0, 0, 9, 0, 0, 0, 0],
        [0, 9, 0, 0, 1, 0, 0, 2, 0],
        [0, 3, 8, 0, 0, 0, 9, 4, 0],
        [4, 0, 0, 0, 0, 0, 0, 0, 2],
        [0, 0, 3, 5, 0, 9, 6, 0, 0],
        [0, 0, 2, 4, 0, 1, 3, 0, 0],
    ]
)

hard_no_106 = np.array(
    [
        [0, 0, 0, 4, 0, 7, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 7, 0, 0],
        [4, 0, 0, 0, 0, 0, 0, 0, 3],
        [0, 2, 0, 3, 0, 9, 0, 4, 0],
        [0, 4, 0, 0, 1, 0, 0, 9, 0],
        [0, 0, 6, 0, 0, 0, 8, 0, 0],
        [5, 0, 0, 0, 0, 0, 0, 0, 8],
        [0, 8, 4, 0, 6, 0, 5, 3, 0],
        [3, 0, 0, 0, 0, 0, 0, 0, 2],
    ]
)

sd1 = np.array(
    [
        [7, 9, 0, 0, 0, 0, 0, 0, 3],
        [0, 0, 0, 0, 0, 0, 0, 6, 0],
        [8, 0, 1, 0, 0, 4, 0, 0, 2],
        [0, 0, 5, 0, 0, 0, 0, 0, 0],
        [3, 0, 0, 1, 0, 0, 0, 0, 0],
        [0, 4, 0, 0, 0, 6, 2, 0, 9],
        [2, 0, 0, 0, 3, 0, 0, 0, 6],
        [0, 3, 0, 6, 0, 5, 4, 2, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
    ]
)

problem = SudokuProblem(initial_board=hard_no_77)

algorithm = GA(
    pop_size=150,
    sampling=MySampling(),
    crossover=RowCrossover(row_cross_rate=0.1, prob=0.2),
    mutation=SwapReinitMutation(swap_rate=0.3, reinit_rate=0.05),
    repair=LocalSearchRepairV2(greedier=True, exhaustive=True, strict=False),
    survival=EPLSurvival(
        n_elite=50,
        sampling=MySampling(),
        swap_rate=0.3,
        reinit_rate=0.05,
        nudge=True,
        force_unique=True,
    ),
    eliminate_duplicates=True,
)

# Wang et al. limit to 10,000 gens for fair comparison
termination = TerminationCollection(
    get_termination("n_gen", 1000),
    ZeroFunctionValueTermination(),
)

start_time = time.perf_counter()
res = minimize(problem, algorithm, termination, seed=0, verbose=True)
end_time = time.perf_counter()

n_gen  |  n_eval  |     f_avg     |     f_min    
     1 |      150 |  1.556667E+01 |  1.000000E+01
     2 |      300 |  1.449333E+01 |             6
     3 |      450 |  1.351333E+01 |             6
     4 |      600 |  1.378000E+01 |             5
     5 |      750 |  1.222000E+01 |             4
     6 |      900 |  1.180000E+01 |             2
     7 |     1050 |  1.172667E+01 |             2
     8 |     1200 |  1.156000E+01 |             2
     9 |     1350 |  9.6666666667 |             2
    10 |     1500 |  1.016000E+01 |             2
    11 |     1650 |  8.7133333333 |             2
    12 |     1800 |  9.4400000000 |             2
    13 |     1950 |  8.6533333333 |             2
    14 |     2100 |  9.3600000000 |             2
    15 |     2250 |  8.9866666667 |             2
    16 |     2400 |  8.8133333333 |             2
    17 |     2550 |  9.0800000000 |             2
    18 |     2700 |  9.1133333333 |             2
    19 |     2850 |  9.1466666667 |             2


In [15]:
res.X.reshape(9, 9)

array([[5, 8, 7, 1, 6, 2, 4, 3, 9],
       [9, 2, 4, 8, 3, 5, 1, 7, 6],
       [3, 6, 1, 9, 4, 7, 2, 8, 5],
       [1, 4, 5, 2, 9, 8, 7, 6, 3],
       [7, 9, 6, 3, 1, 4, 5, 2, 8],
       [2, 3, 8, 7, 5, 6, 9, 4, 1],
       [4, 1, 9, 6, 7, 3, 8, 5, 2],
       [8, 7, 3, 5, 2, 9, 6, 1, 4],
       [6, 5, 2, 4, 8, 1, 3, 9, 7]], dtype=int32)

In [16]:
for i in res.pop[:50]:
    print(i.X.reshape(9, 9))

[[5 8 7 1 6 2 4 3 9]
 [9 2 4 8 3 5 1 7 6]
 [3 6 1 9 4 7 2 8 5]
 [1 4 5 2 9 8 7 6 3]
 [7 9 6 3 1 4 5 2 8]
 [2 3 8 7 5 6 9 4 1]
 [4 1 9 6 7 3 8 5 2]
 [8 7 3 5 2 9 6 1 4]
 [6 5 2 4 8 1 3 9 7]]
[[5 7 8 1 6 2 4 3 9]
 [9 4 1 8 3 5 2 7 6]
 [3 2 6 9 4 7 1 8 5]
 [2 5 7 3 9 4 8 6 1]
 [6 9 4 7 1 8 5 2 3]
 [1 3 8 2 5 6 9 4 7]
 [4 1 5 6 8 3 7 9 2]
 [7 8 3 5 2 9 6 1 4]
 [9 6 2 4 7 1 3 5 8]]
[[5 8 7 1 6 2 4 3 9]
 [9 4 1 8 3 5 2 7 6]
 [3 2 6 9 4 7 1 8 5]
 [2 5 7 3 9 4 8 6 1]
 [6 9 4 7 1 8 5 2 3]
 [1 3 8 2 5 6 9 4 7]
 [4 1 5 6 8 3 7 9 2]
 [8 7 3 5 2 9 6 1 4]
 [9 6 2 4 7 1 3 5 8]]
[[5 8 7 1 6 2 4 3 9]
 [9 4 1 8 3 5 2 7 6]
 [3 2 6 9 4 7 1 8 5]
 [7 2 5 3 9 4 8 6 1]
 [6 9 4 7 1 8 5 2 3]
 [1 3 8 2 5 6 9 4 7]
 [4 1 9 6 8 3 7 5 2]
 [8 7 3 5 2 9 6 1 4]
 [5 6 2 4 7 1 3 9 8]]
[[5 8 7 1 6 2 4 3 9]
 [9 4 2 8 3 5 1 7 6]
 [3 1 6 9 4 7 2 8 5]
 [7 2 5 3 9 4 8 6 1]
 [6 9 4 7 1 8 5 2 3]
 [1 3 8 2 5 6 9 4 7]
 [4 5 1 6 8 3 7 9 2]
 [8 7 3 5 2 9 6 1 4]
 [9 6 2 4 7 1 3 5 8]]
[[5 7 8 1 6 2 4 3 9]
 [9 4 1 8 3 5 2 7 6]
 [3 2 6 

In [17]:
for i in res.pop[::15]:
    print(i.F)
    print(i.X.reshape(9, 9))

[0]
[[5 8 7 1 6 2 4 3 9]
 [9 2 4 8 3 5 1 7 6]
 [3 6 1 9 4 7 2 8 5]
 [1 4 5 2 9 8 7 6 3]
 [7 9 6 3 1 4 5 2 8]
 [2 3 8 7 5 6 9 4 1]
 [4 1 9 6 7 3 8 5 2]
 [8 7 3 5 2 9 6 1 4]
 [6 5 2 4 8 1 3 9 7]]
[2]
[[5 8 7 1 6 2 4 3 9]
 [9 4 1 8 3 5 2 7 6]
 [3 2 6 9 4 7 1 8 5]
 [2 5 7 3 9 4 8 6 1]
 [6 9 4 7 1 8 5 2 3]
 [1 3 8 2 5 6 9 4 7]
 [4 1 9 6 8 3 7 5 2]
 [8 7 3 5 2 9 6 1 4]
 [5 6 2 4 7 1 3 9 8]]
[2]
[[5 8 7 1 6 2 4 3 9]
 [9 2 4 8 3 5 1 7 6]
 [3 1 6 9 4 7 2 8 5]
 [1 4 5 2 9 8 7 6 3]
 [7 9 6 3 1 4 5 2 8]
 [2 3 8 7 5 6 9 4 1]
 [4 1 9 6 7 3 8 5 2]
 [8 7 3 5 2 9 6 1 4]
 [6 5 2 4 8 1 3 9 7]]
[2]
[[5 8 6 1 4 2 7 3 9]
 [9 4 7 8 3 5 2 1 6]
 [3 2 1 9 6 7 4 8 5]
 [1 2 5 7 9 4 8 6 3]
 [6 9 4 3 1 8 5 2 7]
 [7 3 8 2 5 6 9 4 1]
 [4 7 9 6 8 3 1 5 2]
 [8 1 3 5 2 9 6 7 4]
 [5 6 2 4 7 1 3 9 8]]
[4]
[[5 8 6 1 4 2 7 3 9]
 [9 4 7 8 3 5 2 1 6]
 [3 2 1 9 6 7 4 8 5]
 [2 7 5 3 9 4 8 6 1]
 [6 9 4 7 1 8 5 2 3]
 [1 3 8 2 6 5 9 4 7]
 [4 7 9 6 8 3 1 5 2]
 [8 1 3 5 2 9 6 7 4]
 [5 6 2 4 7 1 3 9 8]]
[5]
[[5 8 6 1 4 2 7 3 9]
 [9 2

In [ ]:
def analyze_population_old(res):
    X = res.pop.get("X")
    n_pop = len(X)
    unique_X, counts = np.unique(X, axis=0, return_counts=True)
    n_unique = len(unique_X)
    n_duplicates = n_pop - n_unique

    print(f"--- Population Diversity Analysis ---")
    print(f"Total Population Size: {n_pop}")
    print(f"Unique Individuals:    {n_unique}")
    print(f"Duplicate Count:       {n_duplicates}")
    print(f"Percentage Unique:     {(n_unique / n_pop) * 100:.2f}%")

    if n_unique < n_pop:
        print(f"\nMost common individual appears {np.max(counts)} times.")

    F = res.pop.get("F")
    unique_F = np.unique(F)
    print(f"Unique Fitness Values: {len(unique_F)}")
    print(f"Fitness range:         [{np.min(F)}, {np.max(F)}]")

In [ ]:
analyze_population_old(res)

--- Population Diversity Analysis ---
Total Population Size: 150
Unique Individuals:    150
Duplicate Count:       0
Percentage Unique:     100.00%
Unique Fitness Values: 11
Fitness range:         [0, 18]


In [ ]:
def analyze_population(res):
    pop = res.pop
    X = pop.get("X")
    F = pop.get("F")[:, 0]

    print(f"Best fitness: {res.F[0]}")
    print(f"Fitnesses: {np.unique(F)}")
    print(f"Average fitness: {np.mean(F)}")

    unique_X = np.unique(X, axis=0)
    print(f"Unique grids: {len(unique_X)} / {len(X)}")
    return unique_X

In [25]:
grids = analyze_population(res)

Best fitness: 0
Fitnesses: [ 0  2  4  5  6  7  8  9 10 17 18]
Average fitness: 8.266666666666667
Unique grids: 150 / 150


In [26]:
from collections import Counter

import numpy as np


def fitness_distribution(res):
    F = res.pop.get("F")[:, 0].astype(int)
    counts = Counter(F)
    sorted_fitness = sorted(counts.items())

    print(f"{'Fitness Level':<15} | {'Count':<10} | {'Percentage':<10}")
    print("-" * 45)

    total = len(F)
    for fitness, count in sorted_fitness:
        percentage = (count / total) * 100
        print(f"{fitness:<15} | {count:<10} | {percentage:>8.2f}%")


fitness_distribution(res)

Fitness Level   | Count      | Percentage
---------------------------------------------
0               | 1          |     0.67%
2               | 52         |    34.67%
4               | 12         |     8.00%
5               | 11         |     7.33%
6               | 20         |    13.33%
7               | 1          |     0.67%
8               | 2          |     1.33%
9               | 1          |     0.67%
10              | 2          |     1.33%
17              | 3          |     2.00%
18              | 45         |    30.00%
